<a href="https://colab.research.google.com/github/monishvarghesejoshy/emanzon/blob/adf_publish/databricks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Databricks notebook source
display(dbutils.fs.ls('abfss://emanzonsource@emanzonsa.dfs.core.windows.net/'))

# COMMAND ----------

# MAGIC %sql
# MAGIC create catalog man_cata

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE SCHEMA man_cata.man_schema

# COMMAND ----------

# MAGIC %sql
# MAGIC create table man_cata.man_schema.man_table
# MAGIC (id int,
# MAGIC name string,
# MAGIC updated timestamp)

# COMMAND ----------

df=spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("abfss://emanzonsource@emanzonsa.dfs.core.windows.net/Sales.csv")

# COMMAND ----------

df.display()

# COMMAND ----------

from pyspark.sql import *

df.select(
    col('Item_Weight').alias('Item_Weight'),
    col('Item_Fat_Content').alias('Item_Fat_Content')
).display()

# COMMAND ----------

df.printSchema()

# COMMAND ----------

df1=df.filter((col('Item_Weight') == 0) | (col('Item_Fat_Content') == 'nan'))

# COMMAND ----------


df.groupBy('Outlet_Identifier').sum('Item_Outlet_Sales').alias('sales').display()

# COMMAND ----------

from pyspark.sql import *

df.na.drop().sort('Item_Outlet_Sales', ascending=False).display()

# COMMAND ----------

from pyspark.sql import functions as f
from pyspark.sql.functions import *
df.select('item_type', f.when(col('item_type'
)!= 'a', 1)\
 .otherwise(0)).display()

# COMMAND ----------

# MAGIC %sql
# MAGIC SHOW CATALOGS;
# MAGIC

# COMMAND ----------

# MAGIC %sql
# MAGIC SHOW TABLES;
# MAGIC

# COMMAND ----------

# MAGIC %sql
# MAGIC SHOW METASTORES;
# MAGIC

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE DATABASE SALESDB;

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE SALESDB.MANTB
# MAGIC (
# MAGIC   ID INT,
# MAGIC   NAME STRING
# MAGIC )
# MAGIC

# COMMAND ----------

# MAGIC %sql
# MAGIC INSERT INTO SALESDB.MANTB
# MAGIC SELECT 1, 'MJ' UNION ALL
# MAGIC SELECT 2, 'JJ'

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT * FROM SALESDB.MANTB

# COMMAND ----------

# MAGIC %sql
# MAGIC cREATE EXTERNAL TABLE SALESDB.exttbl
# MAGIC (id int
# MAGIC , name string)
# MAGIC using delta LOCATION 'abfss://emanzondest@emanzonsa.dfs.core.windows.net/salesdb/exttbl';

# COMMAND ----------

# MAGIC %sql
# MAGIC  insert into SALESDB.exttbl
# MAGIC  select * from emanzonws.SALESDB.MANTB

# COMMAND ----------

# MAGIC %sql
# MAGIC  insert into SALESDB.exttbl
# MAGIC  select 3, 'ff' union all
# MAGIC  select 4, 'ee'

# COMMAND ----------

# MAGIC %sql
# MAGIC select * from SALESDB.exttbl

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE SALESDB.exttbl
# MAGIC USING DELTA
# MAGIC LOCATION 'abfss://emanzondest@emanzonsa.dfs.core.windows.net/salesdb/exttbl';

# COMMAND ----------

# MAGIC %sql
# MAGIC DESCRIBE HISTORY SALESDB.exttbl

# COMMAND ----------

# MAGIC %sql
# MAGIC RESTORE TABLE SALESDB.exttbl VERSION AS OF 1;

# COMMAND ----------

# MAGIC %sql
# MAGIC select * from SALESDB.exttbl

# COMMAND ----------

# MAGIC %sql
# MAGIC RESTORE TABLE SALESDB.exttbl VERSION AS OF 2;

# COMMAND ----------

# MAGIC %sql
# MAGIC VACUUM SALESDB.exttbl RETAIN 2 HOURS;

# COMMAND ----------

# MAGIC %sql
# MAGIC create catalog ext_cat
# MAGIC MANAGED LOCATION 'abfss://mycontainer@emanzonsa.dfs.core.windows.net/external_Catalog'

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE schema ext_cat.man_schema

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE ext_cat.man_schema.man_table
# MAGIC (
# MAGIC   id int,
# MAGIC   name STRING
# MAGIC )
# MAGIC USING DELTA

# COMMAND ----------

# MAGIC %sql
# MAGIC create schema ext_cat.ext_schema
# MAGIC MANAGED LOCATION 'abfss://mycontainer@emanzonsa.dfs.core.windows.net/external_schema'

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE ext_cat.ext_schema.man_table
# MAGIC (
# MAGIC   id int,
# MAGIC   name STRING
# MAGIC )
# MAGIC USING DELTA

# COMMAND ----------



# COMMAND ----------

# MAGIC %md
# MAGIC # STREAMING- AUTO **LOADER**

# COMMAND ----------


df=spark.readStream.format('cloudFiles')\
        .option('cloudFiles.format', 'parquet')\
        .option('cloudFiles.schemaLocation', 'abfss://streamdest@emanzonsa.dfs.core.windows.net/checkpoint')\
        .load('abfss://streamsource@emanzonsa.dfs.core.windows.net')

# COMMAND ----------

df.writeStream.format('delta')\
    .option("checkpointLocation", 'abfss://streamdest@emanzonsa.dfs.core.windows.net/checkpoint')\
    .trigger(processingTime='5 seconds')\
        .start('abfss://streamdest@emanzonsa.dfs.core.windows.net/data')

# COMMAND ----------

# MAGIC %sql
# MAGIC create database emanzon_dev

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE emanzon_dev.reports(
# MAGIC 	id int NOT NULL,
# MAGIC 	task_id int NOT NULL,
# MAGIC 	candidate varchar(40) NOT NULL,
# MAGIC 	score int NOT NULL,
# MAGIC 	updated TIMESTAMP
# MAGIC )

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE emanzon_dev.players(
# MAGIC 	player_id int NOT NULL,
# MAGIC 	group_id int NOT NULL,
# MAGIC 	updated TIMESTAMP
# MAGIC )

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE emanzon_dev.matches(
# MAGIC 	match_id int NOT NULL,
# MAGIC 	first_player int NOT NULL,
# MAGIC 	second_player int NOT NULL,
# MAGIC 	first_score int NOT NULL,
# MAGIC 	second_score int NOT NULL,
# MAGIC 	updated TIMESTAMP
# MAGIC )

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE emanzon_db.lookup_employee(
# MAGIC     EmployeeID STRING,
# MAGIC     DepartNo STRING,
# MAGIC     EmployeeName STRING,
# MAGIC     employeeaddress STRING,
# MAGIC     modiffied TIMESTAMP
# MAGIC );
# MAGIC
# MAGIC CREATE TABLE emanzon_db.lookup_department(
# MAGIC     Department_id STRING,
# MAGIC     DepartmentName STRING,
# MAGIC     modiffied TIMESTAMP
# MAGIC );
# MAGIC
# MAGIC CREATE TABLE emanzon_db.OrderDetails(
# MAGIC     OrderID INT,
# MAGIC     OrderRowNo INT,
# MAGIC     OrderValue INT,
# MAGIC     modiffied TIMESTAMP
# MAGIC );

# COMMAND ----------

#broadcast_variables
lookup={'USA': 'United States of America', 'AUS': 'Australia'}
br_var=spark.sparkcontext.broadcast(br_var)
print(br_var.value)

# COMMAND ----------

#broadcast_join
from pyspark.sql.functions import broadcast

# Small lookup dataset
lookup_df = spark.createDataFrame(
    [("USA", "United States"), ("IND", "India"), ("AUS", "Australia")],
    ["code", "country"]
)

# Large dataset
data_df = spark.createDataFrame(
    [("USA", 100), ("IND", 200), ("CAN", 300)],
    ["code", "value"]
)

# Broadcast the small dataset in a join
result = data_df.join(broadcast(lookup_df), on="code", how="left")
result.show()


# COMMAND ----------

from pyspark import StorageLevel

# Example DataFrame
df = spark.range(1, 1000000)

# Persist the DataFrame with MEMORY_AND_DISK
df.persist(StorageLevel.MEMORY_AND_DISK)

# Perform an action to materialize the persistence
df.count()


# COMMAND ----------

#lazy evaluation, executes only last statement
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("LazyEvaluationExample").getOrCreate()

# Sample data
data = [1, 2, 3, 4, 5]
rdd = spark.sparkContext.parallelize(data)

# Transformation: Multiply each element by 2 (not executed yet)
transformed_rdd = rdd.map(lambda x: x * 2)

# Another Transformation: Filter elements greater than 5 (still not executed)
filtered_rdd = transformed_rdd.filter(lambda x: x > 5)

# Action: Collect results (triggers execution)
result = filtered_rdd.collect()
print(result)
